In [ ]:

import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
char_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-characterization-file.csv'


In [ ]:
char_file = pd.read_csv(char_file_path)


In [ ]:
char_file.columns

In [ ]:
python_1_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/usuarios-uncode-python-2021.csv'
python_2_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/usuarios-uncode-python-2023.csv'


In [ ]:
python_1 = pd.read_csv(python_1_path)
python_2 = pd.read_csv(python_2_path)


In [ ]:
python_1 = python_1[['email', 'username']]
python_2 = python_2[['email', 'username']]

In [ ]:
python_files = pd.concat([python_1, python_2])

In [ ]:
len(python_files)

Se borran de los python files los que tengan un correo en NaN porque no nos sirven para contrastar

In [ ]:
python_files['email'] = python_files['email'].str.strip().str.lower()
#python_files['id'] = python_files['id'].str.strip().str.lower()
#python_files['_id'] = python_files['_id'].str.strip().str.lower()
python_files['username'] = python_files['username'].str.strip().str.lower()

In [ ]:
# dropna filtrando solo por la columna 'email'
python_files = python_files.dropna(subset=['email'])

In [ ]:
len(python_files)

In [ ]:
duplicados_dobles = python_files[python_files.duplicated(subset=['email', 'username'], keep=False)]
print(duplicados_dobles.sort_values(by=['email', 'username']))

In [ ]:
python_files = python_files.drop_duplicates(subset=['email', 'username'], keep='first')

In [ ]:
len(python_files)

In [ ]:
char_file['email'] = char_file['email'].str.strip().str.lower()
char_file['uncode_usr'] = char_file['uncode_usr'].str.strip().str.lower()
char_file['username'] = char_file['username'].str.strip().str.lower()


In [ ]:
char_file['uncode_usr']

In [ ]:
len(char_file)

In [ ]:
df_completando_uncode_usr = char_file.merge(
    python_files,
    on='email',
    how='left',
    indicator=True
)


Presuntamente puedo recuperar 1345 usernames

In [ ]:
len(df_completando_uncode_usr)

In [ ]:
df_completando_uncode_usr.head()

In [ ]:

df_completando_uncode_usr['final_uncode_usr'] = df_completando_uncode_usr['uncode_usr'].combine_first(df_completando_uncode_usr['username_y'])

# Un registro se recuperó si uncode_usr era NaN pero final_uncode_usr tiene un valor
recuperados = df_completando_uncode_usr['uncode_usr'].isna() & df_completando_uncode_usr['final_uncode_usr'].notna()
total_recuperados = recuperados.sum()

# 3. Reporte detallado
print("--- Reporte de Recuperación ---")
print(f"Registros recuperados (eran NaN y ahora tienen valor): {total_recuperados}")
print(f"Registros que ya tenían valor en uncode_usr: {df_completando_uncode_usr['uncode_usr'].notna().sum()}")
print(f"Registros que siguen siendo NaN (ambos eran nulos): {df_completando_uncode_usr['final_uncode_usr'].isna().sum()}")

In [ ]:
df_completando_uncode_usr.columns

In [ ]:
df_completando_uncode_usr = df_completando_uncode_usr.rename(columns={'username_x': 'username'})

In [ ]:
df_completando_uncode_usr = df_completando_uncode_usr[['username', 'dondeconocioelcurso', 'encuestainicial', 'motivacion',
       'ocupacion', 'lenguajes_programacion', 'nivelprogramacion',
       'dedicacionsemanal', 'razon_realizar_curso', 'otro_medio',
       'otra_motivacion', 'otro_lenguaje', 'otra_ocupacion', 'email', 'name',
       'platform', 'final_uncode_usr']]

In [ ]:
df_completando_uncode_usr = df_completando_uncode_usr.rename(columns={'final_uncode_usr': 'uncode_usr'})

In [ ]:
df_completando_uncode_usr.columns

Ya tenemos en uncode_usr algunos usuarios recuperados.
Ahora, como existe el username y el uncode_usr, se va a crear una tercera columna ya que en algunos casos en el submission file se encuentra el username o el uncode. Nos quedaremos con el que traiga información

In [ ]:
sub_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-submissions-file.csv'


In [ ]:
sub_file = pd.read_csv(sub_file_path)


In [ ]:
sub_file['username'] = sub_file['username'].str.lower()
sub_file['username'] = sub_file['username'].str.replace('[', '')
sub_file['username'] = sub_file['username'].str.replace(']', '')
sub_file['username'] = sub_file['username'].str.replace("'", '')

In [ ]:
usuarios_en_sub_file = set(sub_file['username'])

def decidir_username(row):
    # existe el 'username' en sub_file?
    if row['username'] in usuarios_en_sub_file:
        return row['username']

    # existe el 'uncode_usr' en sub_file?
    if row['uncode_usr'] in usuarios_en_sub_file:
        return row['uncode_usr']

    # Si ninguno está, se elige el 'username'
    return row['username']

df_completando_uncode_usr['final_username'] = df_completando_uncode_usr.apply(decidir_username, axis=1)

In [ ]:
cambios = ((df_completando_uncode_usr['final_username'] == df_completando_uncode_usr['uncode_usr']) &
           (df_completando_uncode_usr['username'] != df_completando_uncode_usr['uncode_usr'])).sum()

print(f"Total de registros ajustados usando uncode_usr: {cambios}")

In [ ]:
df_completando_uncode_usr.columns

In [ ]:
df_completando_uncode_usr.drop(columns=['username'], inplace=True)

df_completando_uncode_usr.rename(columns={'final_username': 'username'}, inplace=True)

In [ ]:
char_file.columns

In [ ]:
df_completando_uncode_usr.columns

In [ ]:
df_completando_uncode_usr.to_csv('/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/import/final-characterization-file-completed.csv', index=False) # index=False prevents writing the index as a separate column